# Régression - Prédiction du Prix_Revente
L'objectif est d'estimer la variable continue `Prix_Revente` en utilisant différentes approches de régression.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
train = pd.read_csv('../data/processed/train.csv')
val = pd.read_csv('../data/processed/val.csv')
test = pd.read_csv('../data/processed/test.csv')

X_train, y_train = train.drop(['Prix_Revente', 'Rapport_Collecte', 'Categorie'], axis=1), train['Prix_Revente']
X_val, y_val = val.drop(['Prix_Revente', 'Rapport_Collecte', 'Categorie'], axis=1), val['Prix_Revente']
X_test, y_test = test.drop(['Prix_Revente', 'Rapport_Collecte', 'Categorie'], axis=1), test['Prix_Revente']

## Comparaison de 3 modèles de régression
Évaluation de Linear Regression, Random Forest Regressor et XGBoost Regressor.


In [3]:
models_reg = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'XGBoost': XGBRegressor(random_state=42)
}

res_reg = {}
for name, model in models_reg.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    res_reg[name] = model

In [4]:
metrics_reg = []
for name, model in models_reg.items():
    preds = model.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, preds))
    r2 = r2_score(y_val, preds)
    metrics_reg.append({'Modèle': name, 'RMSE': rmse, 'R2': r2})
    print(f"{name} -> RMSE: {rmse:.4f} | R2: {r2:.4f}")

df_metrics = pd.DataFrame(metrics_reg)

Linear Regression -> RMSE: 0.2962 | R2: 0.9126
Random Forest -> RMSE: 0.0994 | R2: 0.9901
XGBoost -> RMSE: 0.0986 | R2: 0.9903


## Scatter Plot Réel vs Prédit
Visualisation de l'alignement entre les valeurs réelles et les valeurs prédites par notre meilleur modèle pour diagnostiquer l'hétéroscédasticité.


In [5]:
best_reg_name = df_metrics.loc[df_metrics['RMSE'].idxmin()]['Modèle']
best_reg = models_reg[best_reg_name]
test_preds = best_reg.predict(X_test)

fig_scatter = px.scatter(
    x=y_test, y=test_preds, 
    labels={'x': 'Valeurs Réelles', 'y': 'Valeurs Prédites'},
    title=f"Réel vs Prédit - {best_reg_name} (Test)",
    opacity=0.6,
    height=500
)
fig_scatter.add_shape(type='line', x0=y_test.min(), y0=y_test.min(), x1=y_test.max(), y1=y_test.max(), line=dict(color='red', dash='dash'))
fig_scatter.show()

## Analyse des Résidus
Le tracé des résidus permet de vérifier qu'il n'y a pas de structure ou de pattern résiduel (qui indiquerait une non-linéarité non capturée).


In [6]:
residuals = y_test - test_preds
fig_res = px.scatter(
    x=test_preds, y=residuals,
    labels={'x': 'Valeurs Prédites', 'y': 'Résidus (Réel - Prédit)'},
    title="Analyse des Résidus",
    opacity=0.6,
    height=400
)
fig_res.add_hline(y=0, line_dash="dash", line_color="red")
fig_res.show()

## Feature Importance
Quelles caractéristiques influencent le plus le prix de revente ?


In [7]:
if hasattr(best_reg, 'feature_importances_'):
    importances = best_reg.feature_importances_
    df_imp = pd.DataFrame({'Feature': X_train.columns, 'Importance': importances}).sort_values(by='Importance', ascending=True)
    fig_imp = px.bar(df_imp, x='Importance', y='Feature', orientation='h', title=f"Importance des variables ({best_reg_name})", height=500)
    fig_imp.show()
else:
    print(f"Le modèle {best_reg_name} n'a pas d'attribut feature_importances_")

## Conclusion
L'évaluation des modèles de régression permet de voir que les méthodes ensemblistes (Random Forest, XGBoost) captent mieux les non-linéarités du prix de revente par rapport à la régression linéaire classique. L'analyse des résidus et l'importance des variables confirment que les caractéristiques intrinsèques des matériaux (ex: Poids) ont un poids prépondérant dans l'estimation de leur valeur.
